In [2]:
from pathlib import Path
BASE = Path("/home/lka/HYBRID_SARIMA")
print(BASE.exists(), BASE)

#Import the file to read it
import pandas as pd
disocupati_trimestral = pd.read_csv(BASE / 'data.csv', 
                       index_col='Period', 
                       parse_dates=True)
print(disocupati_trimestral)

True /home/lka/HYBRID_SARIMA
             15-34 M    15-34 F   35-49 M   35-49 F   50-74 M   50-74 F  \
Period                                                                    
2004-Q1    11.720628  17.198032  4.283283  8.776062  4.004764  5.376203   
2004-Q2    11.410664  16.187372  3.654493  7.196378  3.636536  4.515018   
2004-Q3    10.041510  15.682014  3.520125  6.945425  3.394804  3.976243   
2004-Q4    11.918557  17.227696  4.020448  7.582207  3.604743  3.583677   
2005-Q1    12.334781  16.852636  4.316241  7.577546  3.440298  4.204936   
...              ...        ...       ...       ...       ...       ...   
2025-Q2    11.341464  13.635416  5.106017  7.025122  3.776334  3.889283   
2025-Q3     9.628072  11.355385  4.656108  6.239399  3.108541  3.482718   
2025-Q4    10.903735  10.928362  3.989899  5.704370  3.250742  3.129547   
2026-Q1     9.534584  10.900056  4.316380  5.611548  2.984988  3.392216   
2026-Q2    10.244865  11.855438  5.044973  6.388627  3.268385  3.460483

/tmp/ipykernel_200505/3099411105.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  disocupati_trimestral = pd.read_csv(BASE / 'data.csv',


In [3]:
#Dividir el dataset en training, validation y test
series = [
    '15-34 M', '15-34 F',
    '35-49 M', '35-49 F',
    '50-74 M', '50-74 F',
    'Global_Rate'
]

# Ordenar cronológicamente
df = disocupati_trimestral.sort_index()

# División 70% Train, 15% Validation, 15% Test
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = df[series].iloc[:train_end].copy()
val = df[series].iloc[train_end:val_end].copy()
test = df[series].iloc[val_end:].copy()

# Mostrar resultados para cada serie
#for serie in series:
#    print(f"\nSerie: {serie}")
#    print(f"Train: {len(train[serie])} observaciones")
#    print(f"Validation: {len(val[serie])} observaciones")
#    print(f"Test: {len(test[serie])} observaciones")

print(train)
print(val)
print(test)


             15-34 M    15-34 F   35-49 M    35-49 F   50-74 M   50-74 F  \
Period                                                                     
2004-Q1    11.720628  17.198032  4.283283   8.776062  4.004764  5.376203   
2004-Q2    11.410664  16.187372  3.654493   7.196378  3.636536  4.515018   
2004-Q3    10.041510  15.682014  3.520125   6.945425  3.394804  3.976243   
2004-Q4    11.918557  17.227696  4.020448   7.582207  3.604743  3.583677   
2005-Q1    12.334781  16.852636  4.316241   7.577546  3.440298  4.204936   
...              ...        ...       ...        ...       ...       ...   
2018-Q2    17.908050  21.321705  7.759217  10.435680  6.683032  6.283515   
2018-Q3    16.802144  19.343542  6.191831   9.120602  5.538136  5.299045   
2018-Q4    19.322895  22.132355  7.396393  10.673055  6.327076  6.189145   
2019-Q1    19.163606  21.713279  8.111591  11.250865  6.697788  6.034775   
2019-Q2    17.128382  19.347371  7.161324   9.818228  5.738406  5.356958   

           

In [5]:

sarima_best_params={
    "15-34 M":{"order":(0,1,0),"seasonal_order":(0,1,1,4)},
    "15-34 F":{"order":(0,1,0),"seasonal_order":(0,1,1,4)},
    "35-49 M":{"order":(0,1,0),"seasonal_order":(1,1,2,4)},
    "35-49 F":{"order":(0,1,0),"seasonal_order":(0,1,1,4)},
    "50-74 M":{"order":(2,1,1),"seasonal_order":(0,1,1,4)},
    "50-74 F":{"order":(2,1,0),"seasonal_order":(0,1,1,4)},
    "Global_Rate":{"order":(0,1,0),"seasonal_order":(0,1,1,4)}
}


In [ ]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from itertools import product
from pathlib import Path
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

architectures=[(32,),(64,),(128,),(32,32),(64,32),(64,64),(128,32),(128,64),(128,128)]
grid=list(product(architectures,[0.01,0.001],[4,6,8],[100,200],[16,32],[0.1,0.3,0.5]))

class LSTM(nn.Module):
    def __init__(self,neurons,dropout):
        super().__init__()
        self.layers=nn.ModuleList()
        size=1
        for n in neurons:
            self.layers.append(nn.LSTM(size,n,batch_first=True))
            size=n
        self.drop=nn.Dropout(dropout)
        self.fc=nn.Linear(size,1)

    def forward(self,x):
        for layer in self.layers:
            x,_=layer(x)
        return self.fc(self.drop(x[:,-1,:]))

def sequences(data,window,start,end):
    X=np.array([data[i-window:i] for i in range(start,end)],dtype=np.float32)
    y=np.array(data[start:end],dtype=np.float32)
    X=torch.tensor(X.reshape(-1,window,1),device=device)
    y=torch.tensor(y.reshape(-1,1),device=device)
    return X,y

def fit_lstm(res_train,res_val,params,final=False,best_epoch=None):
    neurons,lr,window,epochs,batch,dropout=params
    torch.manual_seed(42)
    scaler=MinMaxScaler()
    scaler.fit(res_train.reshape(-1,1))
    residuals=np.concatenate([res_train,res_val])
    z=scaler.transform(residuals.reshape(-1,1)).ravel()
    split=len(res_train)
    end=len(z) if final else split
    X,y=sequences(z,window,window,end)
    model=LSTM(neurons,dropout).to(device)
    optimizer=torch.optim.Adam(model.parameters(),lr=lr)
    criterion=nn.MSELoss()
    best=np.inf
    best_epoch_found=0
    wait=0
    total_epochs=best_epoch if final else epochs

    for epoch in range(total_epochs):
        model.train()
        idx=torch.randperm(len(X),device=device)

        for ids in idx.split(batch):
            optimizer.zero_grad()
            loss=criterion(model(X[ids]),y[ids])
            loss.backward()
            optimizer.step()

        if not final:
            model.eval()
            with torch.no_grad():
                Xv,yv=sequences(z,window,split,len(z))
                pred=model(Xv).cpu().numpy()
                actual=yv.cpu().numpy()

            pred=scaler.inverse_transform(pred)
            actual=scaler.inverse_transform(actual)
            mse=mean_squared_error(actual,pred)

            if mse<best-1e-10:
                best=mse
                best_epoch_found=epoch+1
                wait=0
            else:
                wait+=1

            if wait>=15:
                break

    if final:
        return model,scaler

    return best,best_epoch_found,epoch+1

grid_results=[]
results=[]
best_params={}
hybrid_predictions={}

for col,params in sarima_best_params.items():
    y_train=train[col]
    y_val=val[col]
    y_train_val=pd.concat([y_train,y_val])
    y_test=test[col]

    sarima_train=SARIMAX(
        y_train.reset_index(drop=True),
        order=params["order"],
        seasonal_order=params["seasonal_order"],
        enforce_stationarity=True,
        enforce_invertibility=True
    ).fit(disp=False)

    burn=sarima_train.loglikelihood_burn
    res_train=sarima_train.resid.to_numpy()[burn:]
    sarima_val=np.asarray(sarima_train.forecast(steps=len(y_val)))
    res_val=y_val.to_numpy()-sarima_val

    best_mse=np.inf
    best_config=None
    best_epoch=0

    for config in grid:
        mse,epoch,trained=fit_lstm(res_train,res_val,config)
        neurons,lr,window,epochs,batch,dropout=config

        grid_results.append([
            col,len(neurons),str(neurons),lr,window,
            epochs,epoch,trained,batch,dropout,mse
        ])

        if mse<best_mse:
            best_mse=mse
            best_config=config
            best_epoch=epoch

    best_params[col]={
        "params":best_config,
        "best_epoch":best_epoch,
        "MSE_Validation":best_mse
    }

    sarima=SARIMAX(
        y_train_val.reset_index(drop=True),
        order=params["order"],
        seasonal_order=params["seasonal_order"],
        enforce_stationarity=True,
        enforce_invertibility=True
    ).fit(disp=False)

    residuals=sarima.resid.to_numpy()[sarima.loglikelihood_burn:]

    lstm,scaler=fit_lstm(
        residuals,
        np.array([]),
        best_config,
        final=True,
        best_epoch=best_epoch
    )

    sarima_pred=np.asarray(sarima.forecast(steps=len(y_test)))
    history=list(residuals)
    lstm_pred=[]

    for i in range(len(y_test)):
        window=best_config[2]
        x=np.array(history[-window:]).reshape(-1,1)
        x=scaler.transform(x)
        x=torch.tensor(x.reshape(1,window,1),dtype=torch.float32,device=device)

        lstm.eval()
        with torch.no_grad():
            pred=lstm(x).cpu().numpy()

        pred=scaler.inverse_transform(pred)[0,0]
        lstm_pred.append(pred)
        history.append(float(y_test.iloc[i])-sarima_pred[i])

    lstm_pred=np.array(lstm_pred)
    hybrid_pred=sarima_pred+lstm_pred
    real=y_test.to_numpy()

    mae=mean_absolute_error(real,hybrid_pred)
    mse=mean_squared_error(real,hybrid_pred)
    rmse=np.sqrt(mse)
    mask=real!=0
    mape=np.mean(np.abs((real[mask]-hybrid_pred[mask])/real[mask]))*100 if mask.any() else np.nan

    values=y_train_val.to_numpy()
    scale=np.mean(np.abs(values[4:]-values[:-4]))
    mase=mae/scale if scale>0 else np.nan
    r2=r2_score(real,hybrid_pred)

    results.append([col,mae,mse,rmse,mape,mase,r2])

    hybrid_predictions[col]=pd.DataFrame({
        "Observed":real,
        "SARIMA":sarima_pred,
        "LSTM_Residual":lstm_pred,
        "Hybrid":hybrid_pred
    },index=y_test.index)

    print(col,"Validation MSE:",round(best_mse,4),"Test MSE:",round(mse,4))

grid_df=pd.DataFrame(grid_results,columns=[
    "Serie","Layers","Neurons","Learning_Rate","Window",
    "Epochs","Best_Epoch","Epochs_Trained","Batch_Size",
    "Dropout","MSE"
])

grid_df=grid_df.sort_values(["Serie","MSE"]).reset_index(drop=True)
grid_df["Rank"]=grid_df.groupby("Serie").cumcount()+1
top10=grid_df[grid_df["Rank"]<=10]

results_df=pd.DataFrame(results,columns=[
    "Serie","MAE","MSE","RMSE","MAPE","MASE","R2"
])

output=Path("results/SARIMA_LSTM_Sequential")
output.mkdir(parents=True,exist_ok=True)

top10.to_csv(output/"top10_sarima_lstm.csv",index=False)
results_df.to_csv(output/"sarima_lstm_test.csv",index=False)

for col,pred in hybrid_predictions.items():
    pred.to_csv(output/f"pred_{col.replace(' ','_')}.csv")

print(results_df.round(4))
print(best_params)



/home/lka/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:566: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  _index = to_datetime(index)
/home/lka/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use on the supported classes of index.
  self._init_dates(dates, freq)
/home/lka/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:566: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  _index = to_datetime(index)
/home/lka/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: An unsupported index was

ValueError: No supported index is available.